# Kaggle-style review rating model (demo with hidden bugs)
Static review only: the CSV files are not included.


In [ ]:
%matplotlib inline
!pip install -q xgboost imbalanced-learn
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
train["user_id"].value_counts().head()

## Features

In [ ]:
full = pd.concat([train, test])
vec = TfidfVectorizer(max_features=5000)
vec.fit(full["review_text"])

In [ ]:
train["city_te"] = train.groupby("city")["target"].transform("mean")
y = train["target"]
X = train.select_dtypes("number")

## Balance the classes and split

In [ ]:
X_res, y_res = SMOTE().fit_resample(X, y)
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=7)

In [ ]:
model = XGBClassifier(n_estimators=500, early_stopping_rounds=20)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)])

In [ ]:
preds = model.predict(X_test)
print("accuracy", accuracy_score(y_test, preds))